<h1>Obraz binarny</h1>

In [65]:
import numpy as np
import matplotlib.pyplot as plt
import math
import random
import matplotlib.animation as animation

<h4>Funkcja do generacji mapy</h4>

In [66]:
def generate_binary_map(n, density):
    total_pixels = n * n
    num_black_pixels = int(total_pixels * density)

    flat_image = np.concatenate([np.ones(num_black_pixels), 
                                 np.zeros(total_pixels - num_black_pixels)])


    np.random.shuffle(flat_image)

    binary_map = flat_image.reshape((n, n))
    
    return binary_map

<h4>Funkcja do wizualizcji map</h4>

In [67]:
def visualize_comparison(map_start, map_end, label):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    axes[0].imshow(map_start, cmap='binary', interpolation='nearest')
    axes[0].set_title("Initial Map", fontsize=15)
    axes[0].axis('off')
    
    axes[1].imshow(map_end, cmap='binary', interpolation='nearest')
    axes[1].set_title(label, fontsize=15)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

<h4>Generacja GIF</h4>

In [68]:
def generate_annealing_gif(images_history, filename, label):

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis('off')

    im = ax.imshow(images_history[0], cmap='binary', interpolation='nearest')
    title = ax.set_title(label, fontsize=14)

    def update(frame_idx):
        im.set_array(images_history[frame_idx])
        return [im, title]

    ani = animation.FuncAnimation(
        fig, 
        update, 
        frames=len(images_history), 
        blit=True 
    )

    writer = animation.PillowWriter()
    ani.save(filename, writer=writer)
    
    plt.close() 


<h4>Funkcji pomocnicze<h4>

In [69]:
def temp_exp(iteration, initial_t, par):
    return initial_t * (par ** iteration)


def calculate_total_energy(image, energy_func):
    total_e = 0
    rows, cols = image.shape
    
    for r in range(rows):
        for c in range(cols):
            total_e += energy_func(image, r, c)


    return total_e 

<h4>Implementacja Algorytmu<h4>

Stany sąsiednie generowane są poprzez zamianę miejscami losowego piksela czarnego z białym, to gwarantuję stałą gęstość. Akceptuje gorsze rozwiązania z prawdopodobieństwem zależnym od malejącej temperatury

In [70]:
def simulated_annealing(initial_image, energy_func, initial_temp, cooling_rate, num_steps):
    current_image = initial_image.copy()
    

    black_coords = list(map(tuple, np.argwhere(current_image == 1)))
    white_coords = list(map(tuple, np.argwhere(current_image == 0)))
    
    current_temp = initial_temp
    current_total_energy = calculate_total_energy(current_image, energy_func)
    
    best_image = current_image.copy()
    best_energy = current_total_energy
    
    energies_history = []
    temps_history = []
    images_history = []
    
    for step in range(num_steps):
        b_idx = random.randint(0, len(black_coords) - 1)
        w_idx = random.randint(0, len(white_coords) - 1)
        
        b_r, b_c = black_coords[b_idx]
        w_r, w_c = white_coords[w_idx]
        
        energy_before = energy_func(current_image, b_r, b_c) + energy_func(current_image, w_r, w_c)
        
        current_image[b_r, b_c] = 0
        current_image[w_r, w_c] = 1
        
        energy_after = energy_func(current_image, b_r, b_c) + energy_func(current_image, w_r, w_c)
        
        delta_e = energy_after - energy_before
        
        accept = False
        if delta_e < 0: accept = True 
        elif current_temp > 0: 
            probability = math.exp(-delta_e / current_temp)
            if random.random() < probability: accept = True
        
        if accept:
            current_total_energy += delta_e
            
            black_coords[b_idx] = (w_r, w_c)
            white_coords[w_idx] = (b_r, b_c)
            
            if current_total_energy < best_energy:
                best_energy = current_total_energy
                best_image = current_image.copy()
        else:
            current_image[b_r, b_c] = 1
            current_image[w_r, w_c] = 0
            
        energies_history.append(current_total_energy)
        temps_history.append(current_temp)
        
        if step % 3000 == 0: images_history.append(current_image.copy())
            
        current_temp = temp_exp(step, initial_temp, cooling_rate)
        
    return best_image, best_energy, energies_history, temps_history, images_history


* **energy_islands_4n:** Funkcja nagradza układ, w którym sąsiadujące piksele mają ten sam kolor (zmniejsza energię o 1), a karze za piksele o różnych kolorach (zwiększa energię o 1). Minimalizacja tej funkcji doprowadzi do pogrupowania czarnych punktów w duże, zwarte plamy.

* **energy_anisotropy_4n:** Funkcja traktuje sąsiadów niesymetrycznie. Nagradza (spadek energii o 5) tych samych sąsiadów w poziomie (po lewej i prawej), ale silnie karze (wzrost energii o 5) za tych samych sąsiadów w pionie (na górze i na dole). Efektem działania algorytmu z tą funkcją będą wyraźne, poziome paski na obrazie.

* **energy_checkerboard_4n:** Odwrotność funkcji klastrującej. Nagradza sąsiadów o różnych kolorach, a karze sąsiadów o tych samych kolorach. System będzie dążył do stworzenia struktury przypominającej szachownicę.

In [71]:
def energy_islands_4n(image, r, c):
    rows, cols = image.shape
    pixel_color = image[r, c]
    
    neighbors = [
        image[(r - 1) % rows, c],
        image[(r + 1) % rows, c], 
        image[r, (c - 1) % cols], 
        image[r, (c + 1) % cols]  
    ]
    
    energy = 0
    for n in neighbors:
        if n == pixel_color: energy -= 1 
        else: energy += 1 
            
    return energy


def energy_anisotropy_4n(image, r, c):
    rows, cols = image.shape
    pixel_color = image[r, c]
    
    top = image[(r - 1) % rows, c]
    bottom = image[(r + 1) % rows, c]
    left = image[r, (c - 1) % cols]
    right = image[r, (c + 1) % cols]
    
    energy = 0
    
    for n in [left, right]:
        if n == pixel_color: energy -= 5 
        else: energy += 5
            
    for n in [top, bottom]:
        if n == pixel_color: energy += 5 
        else: energy -= 5
            
    return energy


def energy_checkerboard_4n(image, r, c):
    rows, cols = image.shape
    pixel_color = image[r, c]
    
    neighbors = [
        image[(r - 1) % rows, c], 
        image[(r + 1) % rows, c], 
        image[r, (c - 1) % cols], 
        image[r, (c + 1) % cols]  
    ]
    
    energy = 0
    for n in neighbors:
        if n != pixel_color: energy -= 1 
        else: energy += 1 
            
    return energy
        

In [76]:
size = 200
density = 0.3
initial_map = generate_binary_map(size, density)

In [73]:
best_img, best_e, energies, temps, history = simulated_annealing(
    initial_image=initial_map, 
    energy_func=energy_islands_4n, 
    initial_temp=100.0, 
    cooling_rate=0.999, 
    num_steps=600000
)

#generate_annealing_gif(history, filename="./GIFs/islands_4n.gif", label="islands_4n")


![Animacja TSP](GIFs/islands_4n.gif).

In [29]:
best_img, best_e, energies, temps, history = simulated_annealing(
    initial_image=initial_map, 
    energy_func=energy_anisotropy_4n, 
    initial_temp=100.0, 
    cooling_rate=0.999, 
    num_steps=600000
)

#generate_annealing_gif(history, filename="./GIFs/anistropy_4n.gif", label="anistropy_4n")



-3160


![Animacja TSP](GIFs/anistropy_4n.gif).

In [30]:
best_img, best_e, energies, temps, history = simulated_annealing(
    initial_image=initial_map, 
    energy_func=energy_checkerboard_4n, 
    initial_temp=100.0, 
    cooling_rate=0.999, 
    num_steps=600000
)
#generate_annealing_gif(history, filename="./GIFs/checkerboard_4n.gif", label="checkerboard_4n")


6088


![Animacja TSP](GIFs/checkerboard_4n.gif).

<h4>Funkcji dla 8-sąsiedstwa</h4>

* **energy_islands_8n:** Piksele stykające się bokami mają silniejszy wpływ na energię (waga +-2), ponieważ fizycznie znajdują się bliżej niż piksele stykające się rogami (waga +-1). Powinna prowadzić do bardziej gładkiego wyniku niż w funkcji energy_islands_4n.

* **energy_diagonal_rain_8n:** Nagradza sąsiadów układających się wzdłuż jednej osi przekątnej (np. lewy-górny do prawy-dolny), a jednocześnie karze za układanie się wzdłuż prostopadłej przekątnej. Skutkuje to tworzeniem wyraźnych, ukośnych pasków na obrazie.

* **energy_foam_8n:** Zamiast liniowego dodawania kar za każdego sąsiada, funkcja ta ocenia globalne zagęszczenie w otoczeniu 3x3 wokół badanego piksela. Aplikuje wysokie kary (wzrost energii) za bycie pikselem z ≤ 2 sąsiadów lub całkowicie otoczonym (8 sąsiadów). Minimalne wartości energii osiągane są dla 3-5 sąsiadów. System dążąc do takiego stanu tworzy struktury tkankę komórkową lub pianę.

* **energy_snake_8n:** W kierunkach ortogonalnych nagradza posiadanie dokładnie dwóch sąsiadów o tym samym kolorze. Jednocześnie systematycznie karze za posiadanie takich samych sąsiadów na przekątnych, co zapobiega zlewaniu się tych linii w grubsze plamy. W efekcie minimalizacji tej energii algorytm wyżarzania generuje labirynty.

In [74]:

def energy_islands_8n(image, r, c):
    rows, cols = image.shape
    pixel_color = image[r, c]
    
    straights = [
        image[(r - 1) % rows, c], 
        image[(r + 1) % rows, c], 
        image[r, (c - 1) % cols], 
        image[r, (c + 1) % cols]  
    ]
    
    diagonals = [
        image[(r - 1) % rows, (c - 1) % cols], 
        image[(r - 1) % rows, (c + 1) % cols], 
        image[(r + 1) % rows, (c - 1) % cols], 
        image[(r + 1) % rows, (c + 1) % cols]  
    ]
    
    energy = 0
    
    for n in straights:
        if n == pixel_color: energy -= 2
        else: energy += 2
            
    for n in diagonals:
        if n == pixel_color: energy -= 1
        else: energy += 1
            
    return energy


def energy_diagonal_rain_8n(image, r, c):
    rows, cols = image.shape
    pixel_color = image[r, c]
    
    good_diags = [
        image[(r - 1) % rows, (c - 1) % cols], 
        image[(r + 1) % rows, (c + 1) % cols]  
    ]
    
    bad_diags = [
        image[(r - 1) % rows, (c + 1) % cols], 
        image[(r + 1) % rows, (c - 1) % cols]  
    ]
    
    energy = 0
    
    for n in good_diags:
        if n == pixel_color: energy -= 2
        else: energy += 2
            
    for n in bad_diags:
        if n == pixel_color: energy += 2
        else: energy -= 2
            
    return energy


def energy_foam_8n(image, r, c):
    rows, cols = image.shape
    pixel_color = image[r, c]
    
    neighbors = [
        image[(r - 1) % rows, c],               
        image[(r + 1) % rows, c],               
        image[r, (c - 1) % cols],               
        image[r, (c + 1) % cols],               
        image[(r - 1) % rows, (c - 1) % cols],  
        image[(r - 1) % rows, (c + 1) % cols],  
        image[(r + 1) % rows, (c - 1) % cols],  
        image[(r + 1) % rows, (c + 1) % cols]   
    ]
    
    same_color_count = sum(1 for n in neighbors if n == pixel_color)
    
    if same_color_count <= 2: return 10

    elif same_color_count in [3, 4, 5]: return -5

    elif same_color_count == 8: return 10

    else: return 0
    

def energy_snake_8n(image, r, c):
    rows, cols = image.shape
    pixel_color = image[r, c]
    
    direct_neighbors = [
        image[(r - 1) % rows, c],
        image[(r + 1) % rows, c], 
        image[r, (c - 1) % cols], 
        image[r, (c + 1) % cols]  
    ]
    
    diagonal_neighbors = [
        image[(r - 1) % rows, (c - 1) % cols], 
        image[(r - 1) % rows, (c + 1) % cols], 
        image[(r + 1) % rows, (c - 1) % cols], 
        image[(r + 1) % rows, (c + 1) % cols]  
    ]
    
    energy = 0
    
    same_direct = sum(1 for n in direct_neighbors if n == pixel_color)
    
    if same_direct == 2: energy -= 6 
    elif same_direct == 1: energy -= 2 
    else: energy += 8 
        

    for n in diagonal_neighbors:
        if n == pixel_color: energy += 3
        else: energy -= 1
            
    return energy

In [81]:
best_img, best_e, energies, temps, history = simulated_annealing(
    initial_image=initial_map, 
    energy_func=energy_islands_8n, 
    initial_temp=500.0, 
    cooling_rate=0.99988, 
    num_steps=1000000
)

#visualize_comparison(initial_map, best_img, label="physical_realism")
#generate_annealing_gif(history, filename="energy_islands_8n.gif", label="islands_8n")


![Animacja](GIFs/energy_islands_8n.gif).

In [82]:
best_img, best_e, energies, temps, history = simulated_annealing(
    initial_image=initial_map, 
    energy_func=energy_diagonal_rain_8n, 
    initial_temp=500.0, 
    cooling_rate=0.99988, 
    num_steps=1000000
)

#visualize_comparison(initial_map, best_img, label="diagonal_rain")
generate_annealing_gif(history, filename="./GIFs/diagonal_rain_8n.gif", label="diagonal_rain_8n")


![Animacja](GIFs/diagonal_rain_8n.gif).

In [83]:
best_img, best_e, energies, temps, history = simulated_annealing(
    initial_image=initial_map, 
    energy_func=energy_foam_8n, 
    initial_temp=500.0, 
    cooling_rate=0.9988, 
    num_steps=1000000
)

#visualize_comparison(initial_map, best_img, label="foam")
generate_annealing_gif(history, filename="./GIFs/foam_8n.gif", label="foam_8n")


![Animacja](GIFs/foam_8n.gif).

In [91]:
best_img, best_e, energies, temps, history = simulated_annealing(
    initial_image=initial_map, 
    energy_func=energy_snake_8n, 
    initial_temp=1000.0, 
    cooling_rate=0.99988, 
    num_steps=600000
)

#visualize_comparison(initial_map, best_img, label="snake")
#generate_annealing_gif(history, filename="./GIFs/snake_8n2.gif", label="snake_8n")

![Animacja](GIFs/snake_8n.gif).

<h4>Funkcji dla 16-sąsiedstwa</h4>

* **energy_foam_16n:** Kara rośnie proporcjonalnie do odchylenia od pożądanej liczby 7 sąsiadów. Zapobiega to powstawaniu zarówno litych bloków (gdzie liczba sąsiadów wynosiłaby 24), jak i pustych przestrzeni. Algorytm formuje struktrę labiryntu.

* **energy_bamboo:** Funkcja bazująca na koncepcji piany, jednak wykorzystująca asymetryczne okno w osi pionowej (dx od -4 do +2). Zaburzenie symetrii sprawia, że piksele silniej "odczuwają" sąsiadów znajdujących się powyżej nich. W połączeniu z dążeniem do optymalnego zagęszczenia na poziomie 7 sąsiadów, algorytm generuje struktury ulegające silnemu wydłużeniu w osi pionowej, wizualnie przypominające rosnące pędy bambusa.

* **energy_fingerprint:** Funkcja nakłada kary i nagrody w zależności od dominującej osi (poziomej lub pionowej). Asymetria okna zapobiega tworzeniu się idealnie prostych, sztucznych linii. Efektem minimalizacji tej energii są  wijące się pasma podobne do linii papilarnych.

* **energy_glass** Głównym pomysłem funkcji jest silne przyciągęcie bezpośrednich sąsiadów (odległość 1, waga -4) oraz delikatnym odpychaniu w drugim promieniu (odległość 2, waga 1).

In [115]:
def energy_foam_16n(image, r, c):
    rows, cols = image.shape
    pixel_color = image[r, c]
    
    same_color_count = 0
    

    for dx in range(-2, 3):
        for dy in range(-2, 3):
            if dx == 0 and dy == 0:
                continue
                
            n_r = (r + dx) % rows
            n_c = (c + dy) % cols
            
            if image[n_r, n_c] == pixel_color:
                same_color_count += 1
                

    target_neighbors = 7
    energy = abs(same_color_count - target_neighbors) * 2
    
    return energy

def energy_bamboo(image, r, c):
    rows, cols = image.shape
    pixel_color = image[r, c]
    
    same_color_count = 0
    

    for dx in range(-4, 3):
        for dy in range(-2, 3):
            if dx == 0 and dy == 0:
                continue
                
            n_r = (r + dx) % rows
            n_c = (c + dy) % cols
            
            if image[n_r, n_c] == pixel_color:
                same_color_count += 1
                

    target_neighbors = 7
    energy = abs(same_color_count - target_neighbors) * 2
    
    return energy



def energy_fingerprint(image, r, c):
    rows, cols = image.shape
    pixel_color = image[r, c]
    energy = 0
    
    for dx in range(-4, 3):
        for dy in range(-4, 3):
            if dx == 0 and dy == 0: continue
                
            n_r = (r + dx) % rows
            n_c = (c + dy) % cols
            

            if abs(dy) > abs(dx): weight = 2   
            elif abs(dx) > abs(dy): weight = -2
            else: weight = 0  
                
            if image[n_r, n_c] == pixel_color: energy -= weight
            else: energy += weight
                
    return energy

def energy_glass(image, r, c):
    rows, cols = image.shape
    pixel_color = image[r, c]
    energy = 0
    
    for dx in range(-4, 3):
        for dy in range(-4, 3):
            if dx == 0 and dy == 0: continue
                
            n_r = (r + dx) % rows
            n_c = (c + dy) % cols
            

            distance = max(abs(dx), abs(dy))
            
            if distance == 1: weight = -4 
            elif distance == 2: weight = 1
            else: weight = 0  
                
            if image[n_r, n_c] == pixel_color: energy -= weight
            else: energy += weight
                
    return energy

In [116]:
size = 200
density = 0.4
initial_map = generate_binary_map(size, density)

best_img, best_e, energies, temps, history = simulated_annealing(
    initial_image=initial_map, 
    energy_func=energy_glass, 
    initial_temp=100.0, 
    cooling_rate=0.99988, 
    num_steps=600000
)
#visualize_comparison(initial_map, best_img, "glass")
generate_annealing_gif(history, filename="./GIFs/glass.gif", label="glass")


![Animacja ](GIFs/glass.gif).

In [108]:
size = 200
density = 0.4
initial_map = generate_binary_map(size, density)

best_img, best_e, energies, temps, history = simulated_annealing(
    initial_image=initial_map, 
    energy_func=energy_fingerprint, 
    initial_temp=100.0, 
    cooling_rate=0.99988, 
    num_steps=600000
)
#visualize_comparison(initial_map, best_img, "fingerprint")
generate_annealing_gif(history, filename="./GIFs/fingerprint.gif", label="fingerprint")


![Animacja ](GIFs/fingerprint.gif).

In [109]:
size = 200
density = 0.4
initial_map = generate_binary_map(size, density)

best_img, best_e, energies, temps, history = simulated_annealing(
    initial_image=initial_map, 
    energy_func=energy_foam_16n, 
    initial_temp=100.0, 
    cooling_rate=0.99988, 
    num_steps=600000
)
#visualize_comparison(initial_map, best_img, "foa2")
generate_annealing_gif(history, filename="./GIFs/foam_16n.gif", label="foam_16n")

![Animacja](GIFs/foam_16n.gif).

In [110]:
size = 200
density = 0.4
initial_map = generate_binary_map(size, density)

best_img, best_e, energies, temps, history = simulated_annealing(
    initial_image=initial_map, 
    energy_func=energy_bamboo, 
    initial_temp=100.0, 
    cooling_rate=0.99988, 
    num_steps=600000
)
#visualize_comparison(initial_map, best_img, "bamboo")
generate_annealing_gif(history, filename="./GIFs/bamboo.gif", label="bamboo")

![Animacja](GIFs/bamboo.gif).

<h1>Wnioski</h1>
<h3>Jak wpływa sąsiedstwo</h3>

* **4-sąsiedztwo**: Obraz ma tendencję do tworzenia struktur silnie ortogonalnych.

* **8-sąsiedztwo**: Dodanie sąsiadów na skosach sprawia, że energia jest obliczana bardziej "koliście". Wyspy stają się bardziej gładkie,. Możliwe jest też tworzenie ukośnych linii w przypadku funkcji anizotropowych.

* **Większe sąsiedztwa**: Zwiększają promień oddziaływania, co prowadzi do tworzenia grubszych i jeszcze bardziej rozmytych/połączonych struktur, ale znacząco zwiększają czas obliczeń.



<h1>Jak różnią się uzyskane wyniki w zależności od wybranej funkcji energii?</h3>


<h3>Funkcja energii jest bezpośrednio odpowiedzialna za "wzór", do którego dąży układ.</h3>

<h1>Jak różnią się uzyskane wyniki w zależności od szybkości spadku temperatury?</h1>
<h3>Szybkość spadku temperatury decyduje o jakości uzyskanego optimum.</h3>

